In [ ]:
import heapq
import math


# ----------------------------------------------------
# Binary Heap Dijkstra Algorithm
# Category: Dijkstra Family
# Method: Binary Heap
# Works for weighted graphs with non-negative weights
# ----------------------------------------------------
def binary_heap_dijkstra(graph, source, target=None):
    """
    Binary Heap implementation of Dijkstra's Algorithm.

    Parameters:
    graph:
        Adjacency list in this format:
        graph[u] = [(v, weight), (v2, weight2)]

    source:
        Starting node.

    target:
        Optional.
        If target is given, the algorithm can stop when target is reached.
        If target is None, it computes shortest paths from source to all nodes.

    Returns:
        Dictionary containing:
        - method
        - distances
        - parent
        - visited_order
        - distance
        - path
    """

    distances = {}

    for node in graph:
        distances[node] = math.inf

    distances[source] = 0

    parent = {source: None}
    visited_order = []
    visited = set()

    priority_queue = []
    heapq.heappush(priority_queue, (0, source))

    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)

        if current_node in visited:
            continue

        visited.add(current_node)
        visited_order.append(current_node)

        if target is not None and current_node == target:
            break

        for neighbor, weight in graph[current_node]:

            if weight < 0:
                raise ValueError("Dijkstra does not work with negative weights.")

            new_distance = current_distance + weight

            if new_distance < distances.get(neighbor, math.inf):
                distances[neighbor] = new_distance
                parent[neighbor] = current_node
                heapq.heappush(priority_queue, (new_distance, neighbor))

    path = []

    if target is not None:
        path = reconstruct_path(parent, source, target)
        final_distance = distances.get(target, math.inf)
    else:
        final_distance = None

    return {
        "method": "Binary Heap Dijkstra",
        "distances": distances,
        "parent": parent,
        "visited_order": visited_order,
        "distance": final_distance,
        "path": path
    }


# ----------------------------------------------------
# Reconstruct shortest path
# ----------------------------------------------------
def reconstruct_path(parent, source, target):
    """
    Rebuilds the shortest path from source to target.
    """

    if target not in parent and source != target:
        return []

    path = []
    current = target

    while current is not None:
        path.append(current)
        current = parent.get(current)

    path.reverse()

    if len(path) > 0 and path[0] == source:
        return path

    return []

In [ ]:
# ----------------------------------------------------
# A* Algorithm
# ----------------------------------------------------
def a_star(graph, source, target):
    def heuristic(node, target):
        return 0

    g_score = {node: math.inf for node in graph}
    g_score[source] = 0

    parent = {source: None}
    visited_order = []
    visited = set()

    priority_queue = []
    heapq.heappush(priority_queue, (0, source))

    while priority_queue:
        current_f, current_node = heapq.heappop(priority_queue)

        if current_node in visited:
            continue

        visited.add(current_node)
        visited_order.append(current_node)

        if current_node == target:
            break

        for neighbor, weight in graph[current_node]:
            new_g = g_score[current_node] + weight

            if new_g < g_score.get(neighbor, math.inf):
                g_score[neighbor] = new_g
                parent[neighbor] = current_node

                f_score = new_g + heuristic(neighbor, target)
                heapq.heappush(priority_queue, (f_score, neighbor))

    path = reconstruct_path(parent, source, target)

    return {
        "method": "A*",
        "distance": g_score.get(target, math.inf),
        "path": path,
        "parent": parent,
        "visited_order": visited_order
    }

In [ ]:
import heapq
import math
import time
from collections import defaultdict

def load_weighted_undirected_graph(path):
    graph = defaultdict(list)

    with open(path, "r") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#") or line.startswith("%"):
                continue

            parts = line.split()

            if len(parts) == 2:
                u, v = parts
                w = 1.0
            else:
                u, v, w = parts[:3]
                w = float(w)

            graph[u].append((v, w))
            graph[v].append((u, w))

    return graph


def reconstruct_bidirectional_path(parent_forward, parent_backward, meeting_node):
    path_forward = []
    node = meeting_node

    while node is not None:
        path_forward.append(node)
        node = parent_forward[node]

    path_forward.reverse()

    path_backward = []
    node = parent_backward[meeting_node]

    while node is not None:
        path_backward.append(node)
        node = parent_backward[node]

    return path_forward + path_backward


def bidirectional_dijkstra(graph, source, target):
    if source == target:
        return 0, [source], 0

    forward_pq = [(0, source)]
    backward_pq = [(0, target)]

    dist_forward = {source: 0}
    dist_backward = {target: 0}

    parent_forward = {source: None}
    parent_backward = {target: None}

    visited_forward = set()
    visited_backward = set()

    best_distance = math.inf
    meeting_node = None
    explored_nodes = 0

    while forward_pq and backward_pq:
        if forward_pq[0][0] + backward_pq[0][0] >= best_distance:
            break

        if forward_pq[0][0] <= backward_pq[0][0]:
            current_dist, u = heapq.heappop(forward_pq)

            if u in visited_forward:
                continue

            visited_forward.add(u)
            explored_nodes += 1

            if u in dist_backward:
                total = dist_forward[u] + dist_backward[u]
                if total < best_distance:
                    best_distance = total
                    meeting_node = u

            for v, weight in graph[u]:
                new_dist = dist_forward[u] + weight

                if new_dist < dist_forward.get(v, math.inf):
                    dist_forward[v] = new_dist
                    parent_forward[v] = u
                    heapq.heappush(forward_pq, (new_dist, v))

        else:
            current_dist, u = heapq.heappop(backward_pq)

            if u in visited_backward:
                continue

            visited_backward.add(u)
            explored_nodes += 1

            if u in dist_forward:
                total = dist_forward[u] + dist_backward[u]
                if total < best_distance:
                    best_distance = total
                    meeting_node = u

            for v, weight in graph[u]:
                new_dist = dist_backward[u] + weight

                if new_dist < dist_backward.get(v, math.inf):
                    dist_backward[v] = new_dist
                    parent_backward[v] = u
                    heapq.heappush(backward_pq, (new_dist, v))

    if meeting_node is None:
        return math.inf, [], explored_nodes

    path = reconstruct_bidirectional_path(parent_forward, parent_backward, meeting_node)
    return best_distance, path, explored_nodes

In [ ]:
def bidirectional_wrapper(graph, source, target):
    distance, path, explored = bidirectional_dijkstra(graph, source, target)

    return {
        "method": "Bidirectional Dijkstra",
        "distance": distance,
        "path": path,
        "visited_order": list(range(explored))
    }

In [ ]:
#Planar graph algorithM
import time
import heapq

def compute_planar_sssp(graph, source_node, target_node=None):
    pq = [(0, source_node)]
    distances = {node: float('infinity') for node in graph}
    distances[source_node] = 0
    parent_map = {source_node: None}
    
    # Track the exact nodes explored during the algorithm's execution
    explored_nodes = []
    visited = set()

    while pq:
        current_dist, current_node = heapq.heappop(pq)
        
        if current_node in visited:
            continue
            
        visited.add(current_node)
        explored_nodes.append(current_node)

        # Early exit: Stop searching if we reached our specific target
        if target_node and current_node == target_node:
            break

        for neighbor, weight in graph[current_node].items():
            if neighbor in visited:
                continue
            distance = current_dist + weight
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                parent_map[neighbor] = current_node
                heapq.heappush(pq, (distance, neighbor))

    return {
        "status": "completed",
        "final_distances": distances,
        "final_tree": parent_map,
        "explored_nodes": explored_nodes
    }


    


In [ ]:
def planar_wrapper(graph, source, target):
    # convert graph from adjacency list to dictionary format
    planar_graph = {}

    for u in graph:
        planar_graph[u] = {}

        for v, weight in graph[u]:
            planar_graph[u][v] = weight

    result = compute_planar_sssp(planar_graph, source, target)

    parent = result["final_tree"]
    distances = result["final_distances"]

    path = []

    if distances.get(target, math.inf) != math.inf:
        current = target

        while current is not None:
            path.append(current)
            current = parent.get(current)

        path.reverse()

    return {
        "method": "Planar Graph Algorithm",
        "distance": distances.get(target, math.inf),
        "path": path,
        "visited_order": result["explored_nodes"]
    }

In [ ]:
#Approximate algorithm                                                                                                                                                import time
import heapq

class LandmarkDistanceOracle:
    def __init__(self, graph, num_landmarks=20):
        self.graph = graph
        self.num_landmarks = num_landmarks
        self.landmarks = []
        self.l_dist = {}
        self.l_path = {}

    def preprocess(self):
        nodes_sorted = sorted(self.graph.keys(), key=lambda n: len(self.graph[n]), reverse=True)
        self.landmarks = nodes_sorted[:self.num_landmarks]
        for l in self.landmarks:
            d, p = self._dijkstra(l)
            self.l_dist[l] = d
            self.l_path[l] = p

    def _dijkstra(self, start):
        pq = [(0, start)]
        distances = {n: float('infinity') for n in self.graph}
        distances[start] = 0
        parents = {start: None}
        visited = set()
        while pq:
            curr_dist, curr_node = heapq.heappop(pq)
            if curr_node in visited: continue
            visited.add(curr_node)
            for neighbor, weight in self.graph[curr_node].items():
                if neighbor in visited: continue
                d = curr_dist + weight
                if d < distances[neighbor]:
                    distances[neighbor] = d
                    parents[neighbor] = curr_node
                    heapq.heappush(pq, (d, neighbor))
        return distances, parents

    def query(self, source, target):
        explored_count = 0
        best_dist = float('infinity')
        best_landmark = None

        for l in self.landmarks:
            explored_count += 1
            d_s_l = self.l_dist[l].get(source, float('infinity'))
            d_l_t = self.l_dist[l].get(target, float('infinity'))
            if d_s_l + d_l_t < best_dist:
                best_dist = d_s_l + d_l_t
                best_landmark = l

        path = []
        if best_landmark and best_dist != float('infinity'):
            path_s_l, curr = [], source
            while curr is not None:
                path_s_l.append(curr)
                curr = self.l_path[best_landmark].get(curr)
            
            path_t_l, curr = [], target
            while curr is not None:
                path_t_l.append(curr)
                curr = self.l_path[best_landmark].get(curr)
            
            if path_s_l and path_t_l:
                path = path_s_l[:-1] + path_t_l[::-1]

        return best_dist, path, explored_count




In [ ]:
def approximate_wrapper(graph, source, target):
    # convert graph from adjacency list to dictionary format
    oracle_graph = {}

    for u in graph:
        oracle_graph[u] = {}

        for v, weight in graph[u]:
            oracle_graph[u][v] = weight

    oracle = LandmarkDistanceOracle(oracle_graph, num_landmarks=20)
    oracle.preprocess()

    distance, path, explored = oracle.query(source, target)

    return {
        "method": "Approximate Landmark Oracle",
        "distance": distance,
        "path": path,
        "visited_order": list(range(explored))
    }

In [ ]:
import heapq
import math
import time
import os
from collections import defaultdict



def ml_dijkstra(graph, source, target=None, allowed_nodes=None):
    pq = [(0, source)]
    dist = {source: 0}
    parent = {source: None}
    visited = set()
    explored_nodes = 0

    while pq:
        current_dist, u = heapq.heappop(pq)

        if u in visited:
            continue

        visited.add(u)
        explored_nodes += 1

        if target is not None and u == target:
            break

        for v, weight in graph[u]:
            if allowed_nodes is not None and v not in allowed_nodes:
                continue

            new_dist = current_dist + weight

            if new_dist < dist.get(v, math.inf):
                dist[v] = new_dist
                parent[v] = u
                heapq.heappush(pq, (new_dist, v))

    if target is None:
        return dist, parent, explored_nodes

    if target not in dist:
        return math.inf, [], explored_nodes

    path = []
    node = target

    while node is not None:
        path.append(node)
        node = parent[node]

    path.reverse()
    return dist[target], path, explored_nodes


def preprocess_multi_level(graph, number_of_regions=6):
    nodes = sorted(graph.keys())
    region_size = math.ceil(len(nodes) / number_of_regions)

    node_region = {}
    regions = defaultdict(set)

    for i, node in enumerate(nodes):
        region = i // region_size
        node_region[node] = region
        regions[region].add(node)

    boundary_nodes = set()
    overlay_graph = defaultdict(list)

    for u in graph:
        for v, weight in graph[u]:
            if node_region[u] != node_region[v]:
                boundary_nodes.add(u)
                boundary_nodes.add(v)
                overlay_graph[u].append((v, weight))

    for region, region_nodes in regions.items():
        region_boundaries = [node for node in region_nodes if node in boundary_nodes]

        for source_boundary in region_boundaries:
            distances, parents, explored = ml_dijkstra(graph, source_boundary, allowed_nodes=region_nodes)

            for target_boundary in region_boundaries:
                if source_boundary != target_boundary and target_boundary in distances:
                    overlay_graph[source_boundary].append((target_boundary, distances[target_boundary]))

    return {
        "node_region": node_region,
        "regions": regions,
        "boundary_nodes": boundary_nodes,
        "overlay_graph": overlay_graph
    }


def multi_level_dijkstra(graph, preprocessed_graph, source, target):
    if source == target:
        return 0, [source], 0

    node_region = preprocessed_graph["node_region"]
    regions = preprocessed_graph["regions"]
    boundary_nodes = preprocessed_graph["boundary_nodes"]
    overlay_graph = preprocessed_graph["overlay_graph"]

    temp_graph = defaultdict(list)

    for u in overlay_graph:
        temp_graph[u].extend(overlay_graph[u])

    source_region = node_region[source]
    target_region = node_region[target]

    source_region_nodes = regions[source_region]
    target_region_nodes = regions[target_region]

    source_distances, source_parents, source_explored = ml_dijkstra(graph, source, allowed_nodes=source_region_nodes)
    target_distances, target_parents, target_explored = ml_dijkstra(graph, target, allowed_nodes=target_region_nodes)

    for b in boundary_nodes:
        if b in source_region_nodes and b in source_distances:
            temp_graph[source].append((b, source_distances[b]))
            temp_graph[b].append((source, source_distances[b]))

        if b in target_region_nodes and b in target_distances:
            temp_graph[target].append((b, target_distances[b]))
            temp_graph[b].append((target, target_distances[b]))

    if target in source_distances:
        temp_graph[source].append((target, source_distances[target]))
        temp_graph[target].append((source, source_distances[target]))

    distance, path, overlay_explored = ml_dijkstra(temp_graph, source, target)

    explored_nodes = source_explored + target_explored + overlay_explored

    return distance, path, explored_nodes

In [ ]:
def multi_level_wrapper(graph, source, target):

    preprocessed = preprocess_multi_level(
        graph,
        number_of_regions=6
    )

    distance, path, explored = multi_level_dijkstra(
        graph,
        preprocessed,
        source,
        target
    )

    return {
        "method": "Multi-Level Dijkstra",
        "distance": distance,
        "path": path,
        "visited_order": list(range(explored))
    }

In [ ]:
from collections import defaultdict

def load_graph(file_path):
    graph = defaultdict(list)
    first_data_line = True

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("%") or line.startswith("#"):
                continue

            parts = line.split()

            # skip Matrix Market size line, example: 332 332 2126
            if first_data_line and file_path.endswith(".mtx") and len(parts) == 3:
                first_data_line = False
                continue

            first_data_line = False

            if len(parts) < 2:
                continue

            u = int(parts[0])
            v = int(parts[1])

            if len(parts) >= 3:
                weight = float(parts[2])
            else:
                weight = 1.0

            graph[u].append((v, weight))
            graph[v].append((u, weight))

    return graph

In [ ]:
dataset_paths = [
    "inf-USAir97.mtx",
    "inf-euroroad.edges","inf-openflights.edges","inf-power.mtx","inf-roadNet-PA.mtx",
]

In [ ]:
# ----------------------------------------------------
# Test all algorithms on all datasets
# ----------------------------------------------------


for file_path in dataset_paths:
    print("\n==============================")
    print("Dataset:", file_path)
    print("==============================")

    graph = load_graph(file_path)

    nodes = list(graph.keys())
    source = nodes[0]

    reachable = binary_heap_dijkstra(graph, source)

    reachable_nodes = [
        node for node, dist in reachable["distances"].items()
        if dist != math.inf
    ]

    target = reachable_nodes[-1]

    print("Source:", source)
    print("Target:", target)

    # Binary Heap
    binary_result = binary_heap_dijkstra(graph, source, target)

    print("\nBinary Heap Dijkstra")
    print("Distance:", binary_result["distance"])
    print("Path length:", len(binary_result["path"]))
    print("Path:", binary_result["path"])
    print("Visited nodes:", len(binary_result["visited_order"]))

    # A*
    astar_result = a_star(graph, source, target)

    print("\nA*")
    print("Distance:", astar_result["distance"])
    print("Path length:", len(astar_result["path"]))
    print("Path:", astar_result["path"])
    print("Visited nodes:", len(astar_result["visited_order"]))

    # Bidirectional Dijkstra
    distance, path, explored = bidirectional_dijkstra(graph, source, target)

    print("\nBidirectional Dijkstra")
    print("Distance:", distance)
    print("Path length:", len(path))
    print("Path:", path)
    print("Visited nodes:", explored)

    # Planar Graph Algorithm
    planar_result = planar_wrapper(graph, source, target)

    print("\nPlanar Graph Algorithm")
    print("Distance:", planar_result["distance"])
    print("Path length:", len(planar_result["path"]))
    print("Path:", planar_result["path"])
    print("Visited nodes:", len(planar_result["visited_order"]))

    # Approximate Landmark Oracle
    approximate_result = approximate_wrapper(graph, source, target)

    print("\nApproximate Landmark Oracle")
    print("Distance:", approximate_result["distance"])
    print("Path length:", len(approximate_result["path"]))
    print("Path:", approximate_result["path"])
    print("Visited nodes:", len(approximate_result["visited_order"]))

    # Multi-Level Dijkstra
    multi_result = multi_level_wrapper(graph, source, target)

    print("\nMulti-Level Dijkstra")
    print("Distance:", multi_result["distance"])
    print("Path length:", len(multi_result["path"]))
    print("Path:", multi_result["path"])
    print("Visited nodes:", len(multi_result["visited_order"]))

In [ ]:
import random
import time

def run_multiple_tests(graph, algorithm, number_of_tests=10):

    nodes = list(graph.keys())
    results = []

    for i in range(number_of_tests):

        source, target = random.sample(nodes, 2)

        start = time.time()
        result = algorithm(graph, source, target)
        end = time.time()

        results.append({
            "test": i + 1,
            "algorithm": result["method"],
            "source": source,
            "target": target,
            "distance": result["distance"],
            "path_length": len(result["path"]),
            "path": result["path"],
            "explored_nodes": len(result["visited_order"]),
            "time_seconds": end - start
        })

    return results

In [ ]:
for dataset in dataset_paths:

    print("\n")
    print("########################################")
    print("DATASET:", dataset)
    print("########################################")

    graph = load_graph(dataset)

    for algorithm in [
       binary_heap_dijkstra,
       a_star,
       bidirectional_wrapper,
       planar_wrapper,
       approximate_wrapper,
       multi_level_wrapper
    ]:

        results = run_multiple_tests(
            graph,
            algorithm,
            10
        )

        print("\n====================")
        print(results[0]["algorithm"])
        print("====================")

        for r in results:
            print(
                f"Test {r['test']} | "
                f"Distance={r['distance']} | "
                f"Path Length={r['path_length']} | "
                f"Time={r['time_seconds']:.6f}"
            )

In [ ]:
import pandas as pd
import math

algorithms = [
    binary_heap_dijkstra,
    a_star,
    bidirectional_wrapper,
    planar_wrapper,
    approximate_wrapper,
    multi_level_wrapper
]

summary_results = []

for dataset in dataset_paths:

    print("Running:", dataset)

    graph = load_graph(dataset)

    for algorithm in algorithms:

        results = run_multiple_tests(
            graph,
            algorithm,
            10
        )

        avg_runtime = sum(
            r["time_seconds"] for r in results
        ) / len(results)

        avg_distance = sum(
            r["distance"]
            for r in results
            if r["distance"] != math.inf
        ) / len([
            r for r in results
            if r["distance"] != math.inf
        ])

        avg_path_length = sum(
            r["path_length"]
            for r in results
        ) / len(results)

        avg_explored_nodes = sum(
            r["explored_nodes"]
            for r in results
        ) / len(results)

        summary_results.append({
            "Dataset": dataset,
            "Algorithm": results[0]["algorithm"],
            "Avg Runtime": avg_runtime,
            "Avg Distance": avg_distance,
            "Avg Path Length": avg_path_length,
            "Avg Explored Nodes": avg_explored_nodes
        })

summary_table = pd.DataFrame(summary_results)

summary_table

In [ ]:
import matplotlib.pyplot as plt

for dataset in dataset_paths:
    subset = summary_table[summary_table["Dataset"] == dataset]

    plt.figure(figsize=(10, 5))
    plt.bar(subset["Algorithm"], subset["Avg Runtime"])
    plt.title(f"Average Runtime Comparison - {dataset}")
    plt.xlabel("Algorithm")
    plt.ylabel("Average Runtime (seconds)")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
dataset_info = []

for dataset in dataset_paths:

    graph = load_graph(dataset)

    num_nodes = len(graph)

    num_edges = sum(
        len(graph[node])
        for node in graph
    ) // 2

    dataset_info.append({
        "Dataset": dataset,
        "Nodes": num_nodes,
        "Edges": num_edges
    })

dataset_info = pd.DataFrame(dataset_info)

dataset_info

In [ ]:
growth_table = pd.merge(
    summary_table,
    dataset_info,
    on="Dataset"
)

growth_table

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

for algorithm in growth_table["Algorithm"].unique():

    subset = growth_table[
        growth_table["Algorithm"] == algorithm
    ]

    subset = subset.sort_values("Nodes")

    plt.plot(
        subset["Nodes"],
        subset["Avg Runtime"],
        marker="o",
        label=algorithm
    )

plt.xlabel("Number of Nodes")
plt.ylabel("Average Runtime (seconds)")
plt.title("Runtime Growth vs Number of Nodes")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

for algorithm in growth_table["Algorithm"].unique():

    subset = growth_table[
        growth_table["Algorithm"] == algorithm
    ]

    subset = subset.sort_values("Edges")

    plt.plot(
        subset["Edges"],
        subset["Avg Runtime"],
        marker="o",
        label=algorithm
    )

plt.xlabel("Number of Edges")
plt.ylabel("Average Runtime (seconds)")
plt.title("Runtime Growth vs Number of Edges")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import time
import os

def create_sssp_gif(graph, algorithm, dataset_name, max_frames=80):
    nodes = list(graph.keys())
    source = nodes[0]

    reachable = binary_heap_dijkstra(graph, source)

    reachable_nodes = [
        node for node, dist in reachable["distances"].items()
        if dist != math.inf
    ]

    target = reachable_nodes[-1]

    start_time = time.time()
    result = algorithm(graph, source, target)
    end_time = time.time()

    execution_time = end_time - start_time

    visited_order = result["visited_order"]
    path = result["path"]

    G = nx.Graph()

    for u in graph:
        for v, w in graph[u]:
            G.add_edge(u, v, weight=w)

    shown_nodes = set(visited_order[:max_frames]).union(set(path))
    G_small = G.subgraph(shown_nodes).copy()

    pos = nx.spring_layout(G_small, seed=42)

    fig, ax = plt.subplots(figsize=(8, 6))

    frames = min(len(visited_order), max_frames)

    def update(frame):
        ax.clear()

        current_nodes = set(visited_order[:frame + 1])
        current_nodes = current_nodes.intersection(G_small.nodes())

        path_edges = list(zip(path, path[1:]))

        nx.draw_networkx_edges(G_small, pos, ax=ax, alpha=0.2)
        nx.draw_networkx_nodes(G_small, pos, ax=ax, node_size=40)

        nx.draw_networkx_nodes(
            G_small,
            pos,
            nodelist=list(current_nodes),
            node_size=80,
            ax=ax
        )

        nx.draw_networkx_edges(
            G_small,
            pos,
            edgelist=[
                edge for edge in path_edges
                if edge[0] in G_small.nodes() and edge[1] in G_small.nodes()
            ],
            width=2,
            ax=ax
        )

        ax.set_title(
            f"{result['method']} on {dataset_name}\n"
            f"Step {frame + 1}/{frames} | Time: {execution_time:.6f}s"
        )
        ax.axis("off")

    animation = FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=300,
        repeat=False
    )

    safe_name = result["method"].replace(" ", "_").replace("*", "star")
    safe_dataset = dataset_name.replace(".", "_").replace("-", "_")

    filename = f"{safe_name}_{safe_dataset}.gif"

    animation.save(filename, writer=PillowWriter(fps=3))
    plt.close()

    print("Saved:", filename)
    print("Execution time:", execution_time)

    return filename

In [ ]:
algorithms = [
    binary_heap_dijkstra,
    a_star,
    bidirectional_wrapper,
    planar_wrapper,
    approximate_wrapper,
    multi_level_wrapper
]

for dataset in dataset_paths:
    graph = load_graph(dataset)

    for algorithm in algorithms:
        create_sssp_gif(
            graph,
            algorithm,
            dataset,
            max_frames=80
        )

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import time
import math
import os


def visualize_sssp_video(graph, algorithm, dataset_name, max_nodes=120, max_frames=120):
    nodes = list(graph.keys())
    source = nodes[0]

    reachable = binary_heap_dijkstra(graph, source)
    reachable_nodes = [
        node for node, dist in reachable["distances"].items()
        if dist != math.inf
    ]
    target = reachable_nodes[-1]

    start_time = time.time()
    result = algorithm(graph, source, target)
    end_time = time.time()

    execution_time = end_time - start_time
    visited_order = result["visited_order"]
    path = result["path"]
    method = result["method"]

    selected_nodes = list(dict.fromkeys(visited_order[:max_nodes] + path))

    G = nx.Graph()
    for u in selected_nodes:
        for v, w in graph[u]:
            if v in selected_nodes:
                G.add_edge(u, v, weight=w)

    pos = nx.spring_layout(G, seed=42)

    path_edges = list(zip(path, path[1:]))

    total_frames = min(len(visited_order), max_frames)

    fig, ax = plt.subplots(figsize=(10, 7))

    def update(frame):
        ax.clear()

        current_nodes = set(visited_order[:frame + 1])
        current_nodes = current_nodes.intersection(set(G.nodes()))

        current_edges = []
        for u in current_nodes:
            for v, w in graph[u]:
                if v in current_nodes and u in G.nodes() and v in G.nodes():
                    current_edges.append((u, v))

        current_path_edges = [
            (u, v) for u, v in path_edges
            if u in current_nodes and v in current_nodes
        ]

        unvisited_nodes = [
            n for n in G.nodes()
            if n not in current_nodes
        ]

        visited_nodes = [
            n for n in G.nodes()
            if n in current_nodes
        ]

        nx.draw_networkx_edges(
            G, pos,
            edgelist=list(G.edges()),
            edge_color="lightgray",
            width=0.6,
            alpha=0.25,
            ax=ax
        )

        nx.draw_networkx_edges(
            G, pos,
            edgelist=current_edges,
            edge_color="gray",
            width=1.0,
            alpha=0.7,
            ax=ax
        )

        nx.draw_networkx_edges(
            G, pos,
            edgelist=current_path_edges,
            edge_color="red",
            width=3,
            ax=ax
        )

        nx.draw_networkx_nodes(
            G, pos,
            nodelist=unvisited_nodes,
            node_color="white",
            edgecolors="black",
            node_size=80,
            ax=ax
        )

        nx.draw_networkx_nodes(
            G, pos,
            nodelist=visited_nodes,
            node_color="dodgerblue",
            edgecolors="black",
            node_size=90,
            ax=ax
        )

        if source in G.nodes():
            nx.draw_networkx_nodes(
                G, pos,
                nodelist=[source],
                node_color="limegreen",
                edgecolors="black",
                node_size=180,
                ax=ax
            )

        if target in G.nodes():
            nx.draw_networkx_nodes(
                G, pos,
                nodelist=[target],
                node_color="orange",
                edgecolors="black",
                node_size=180,
                ax=ax
            )

        current_node = visited_order[frame]

        if current_node in G.nodes():
            nx.draw_networkx_nodes(
                G, pos,
                nodelist=[current_node],
                node_color="yellow",
                edgecolors="black",
                node_size=220,
                ax=ax
            )

        ax.set_title(
            f"{method} | {dataset_name}\n"
            f"Step {frame + 1}/{total_frames} | Source: {source} | Target: {target} | "
            f"Execution Time: {execution_time:.6f}s",
            fontsize=11
        )

        ax.text(
            0.02,
            0.02,
            f"Blue = visited nodes\nYellow = current node\nRed = final path when reached\n"
            f"Path length: {len(path)} | Distance: {result['distance']}",
            transform=ax.transAxes,
            fontsize=9,
            bbox=dict(facecolor="white", alpha=0.8)
        )

        ax.axis("off")

    safe_method = method.replace(" ", "_").replace("*", "star").replace("/", "_")
    safe_dataset = dataset_name.replace(".", "_").replace("-", "_").replace("/", "_")

    filename = f"{safe_method}_{safe_dataset}_sssp.gif"

    animation = FuncAnimation(
        fig,
        update,
        frames=total_frames,
        interval=350,
        repeat=False
    )

    animation.save(filename, writer=PillowWriter(fps=3))
    plt.close()

    print("Saved:", filename)
    print("Execution time:", execution_time)
    return filename

In [ ]:
algorithms = [
    binary_heap_dijkstra,
    a_star,
    bidirectional_wrapper,
    planar_wrapper,
    approximate_wrapper,
    multi_level_wrapper
]

for dataset in dataset_paths:
    graph = load_graph(dataset)

    for algorithm in algorithms:
        visualize_sssp_video(
            graph,
            algorithm,
            dataset,
            max_nodes=120,
            max_frames=120
        )